In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

In [ ]:
# Load the unified dataset
data_path = Path('unified_dataset.jsonl')
df = pd.read_json(data_path, lines=True)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
display(df.head())

In [ ]:
# Schema overview
print("Column dtypes:")
print(df.dtypes)

print("\n" + "="*50)
print("Basic info:")
df.info()

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print("Missing values summary:")
display(missing_df)

# Visualize missingness
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Missing Value Percentage by Column')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution analysis
if 'label' in df.columns and df['label'].notna().sum() > 0:
    label_counts = df['label'].value_counts()
    print(f"Label column: {df['label'].notna().sum()} non-null values out of {len(df)}")
    print(f"Unique labels: {df['label'].nunique()}")
    
    # Show top labels
    print("\nTop 10 labels:")
    display(label_counts.head(10))
    
    # Visualize label distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    label_counts.head(20).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Label Distribution (Top 20)')
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No label data available for distribution analysis.")

In [ ]:
# Text length distribution
if 'text' in df.columns and df['text'].notna().sum() > 0:
    df['text_length'] = df['text'].str.len()
    df['text_word_count'] = df['text'].str.split().str.len()
    
    print("Text length statistics (characters):")
    print(df['text_length'].describe())
    
    print("\nText word count statistics:")
    print(df['text_word_count'].describe())
    
    # Visualize text length distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(df['text_length'], bins=30, color='teal', edgecolor='white')
    axes[0].set_title('Text Length Distribution (Characters)')
    axes[0].set_xlabel('Characters')
    axes[0].set_ylabel('Frequency')
    
    axes[1].hist(df['text_word_count'], bins=30, color='coral', edgecolor='white')
    axes[1].set_title('Text Length Distribution (Words)')
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("No text data available for length analysis.")

In [ ]:
# Source distribution analysis
if 'source' in df.columns:
    source_counts = df['source'].value_counts()
    print("Source distribution:")
    display(source_counts)
    
    # Visualize source distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    source_counts.plot(kind='bar', ax=ax, color='mediumseagreen')
    ax.set_title('Data Source Distribution')
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No source column available.")

In [ ]:
# Metadata exploration
if 'metadata' in df.columns and df['metadata'].notna().sum() > 0:
    # Extract metadata keys
    metadata_keys = df['metadata'].dropna().apply(lambda x: list(x.keys()) if isinstance(x, dict) else []).explode().unique()
    print(f"Metadata keys found: {list(metadata_keys)}")
    
    # Check for cleaning_status if present
    if 'cleaning_status' in metadata_keys:
        cleaning_counts = df['metadata'].dropna().apply(lambda x: x.get('cleaning_status', 'unknown') if isinstance(x, dict) else 'unknown').value_counts()
        print("\nCleaning status distribution:")
        display(cleaning_counts)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        cleaning_counts.plot(kind='bar', ax=ax, color='mediumpurple')
        ax.set_title('Cleaning Status Distribution')
        ax.set_ylabel('Count')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No metadata available for analysis.")

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_non_empty: all 150 rows have valid text content, text_length_distribution: check for outliers (min=105, max=36698), label_format_consistency: verify '#### answer' format in 50 labeled rows, source_distribution: understand data mix (GSM8K=50, Russian=90, Euler=10)
- Lower-value checks: class_imbalance: NOT a classification task - labels are numeric answers, not classes, audio_image_modality_checks: columns are entirely null, drop them, duplicate_detection: no exact or normalized duplicates found
- Priority actions: drop_irrelevant_modality_columns, preserve_unlabeled_rows_with_flag, validate_label_format

### Strategy Justification

This is a math problem-solving dataset for answer generation (NOT classification). The 100 unlabeled rows from project-euler and Russian olympiads contain valid problems with metadata - they should be preserved for inference or future annotation. Audio (100% null) and image (93% null) columns were dropped as irrelevant. Deduplication was performed on normalized text. Outlier clipping applied to text length. Label status flag added for downstream processing.

- Missing values: `median (not applicable - text labels, no numeric imputation needed)`
- Duplicates: `drop (normalized text deduplication)`
- Outliers: `clip_iqr (applied to text length)`

### Findings

- Missing values before cleaning: 390
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 0
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if {'source', 'label'}.issubset(raw_df.columns):
    coverage = raw_df.assign(label_present=raw_df['label'].notna()).groupby('source', dropna=False)['label_present'].mean().sort_values(ascending=False).head(10)
    if not coverage.empty:
        sns.barplot(x=coverage.values, y=coverage.index.astype(str), ax=axes[0, 1], color='#2563eb')
        axes[0, 1].set_title('Label coverage by source')
        axes[0, 1].set_xlim(0, 1)
    else:
        axes[0, 1].text(0.5, 0.5, 'No source coverage data', ha='center', va='center')
        axes[0, 1].set_axis_off()
elif 'source' in raw_df.columns:
    source_counts = raw_df['source'].astype(str).value_counts().head(10)
    sns.barplot(x=source_counts.values, y=source_counts.index, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Top sources')
else:
    axes[0, 1].text(0.5, 0.5, 'No source column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
